# 09 — Cross-city EDA: Chicago vs NYC vs LA

**Owner:** Bella · **Date:** 2026-07-18 · **Workstream:** Modeling (cross-city)

The platform now serves three cities from three separate feeds. They do **not**
share a label or a raw feature encoding (Chicago = pass/fail + priority codes +
free text; NYC = 0-N score + letter grade + DOHMH enforcement action; LA = 0-100
score + letter grade + service type). This notebook profiles what's the same and
what's different, and shows *why the same modeling recipe lands differently* in
each city. It ties directly to the 2026-07-18 feature work (see
`docs/model-experiments.md`): NYC gained from its closure signal, LA did not.

Charts are saved to `reports/figures/cross_city/` for the PR.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import build_nyc_scores as nyc
import build_la_scores as la
from foodsafety.config import FEATURES_PATH

FIG = ROOT / "reports" / "figures" / "cross_city"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
COL = {"Chicago": "#2166AC", "NYC": "#4D9221", "LA": "#B2182B"}
print("setup ok")

## Build one comparable frame per city
Each city keeps its own label and feature names; we standardize a few axes (current-bad flag, prior bad-rate, inspection month) so the charts line up.

In [ ]:
chi = pd.read_parquet(FEATURES_PATH)
if "right_truncated" in chi.columns:
    chi = chi[~chi["right_truncated"]].copy()
chi["month"] = pd.to_datetime(chi["inspection_date"]).dt.month
chi["prior_rate"] = chi["prior_fails"] / chi["prior_inspections"].replace(0, np.nan)

nev = nyc.build_events()[0]
nev = nev[nev["next_score"].notna()].copy()
nev["month"] = nev["inspection_date"].dt.month

lraw = la.build_raw()
lev = la.build_events(lraw)[0]
lev = lev[lev["next_score"].notna()].copy()
lev["month"] = lev["inspection_date"].dt.month

CITY = {
    "Chicago": dict(df=chi, label="y_fail_or_critical_next_180d", cur="was_fail", prior="prior_rate"),
    "NYC": dict(df=nev, label="y_next_bc", cur="cur_is_bad", prior="prior_bad_rate"),
    "LA": dict(df=lev, label="y_next_bad", cur="cur_is_bad", prior="prior_bad_rate"),
}
for c, d in CITY.items():
    n = len(d["df"]); base = d["df"][d["label"]].mean()
    print(f"{c:8} rows={n:>7,}  forward-risk base rate={base:.1%}")

### 1. Forward-risk base rate
How common the predicted event is. This is the single biggest driver of how hard the problem is and why raw PR-AUC isn't comparable across cities.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
rates = {c: CITY[c]["df"][CITY[c]["label"]].mean() for c in CITY}
bars = ax.bar(list(rates), [rates[c] * 100 for c in rates], color=[COL[c] for c in rates])
for b, c in zip(bars, rates):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3, f"{rates[c]*100:.1f}%",
            ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Forward-risk positive rate (%)")
ax.set_title("Label prevalence differs 4x across cities")
ax.margins(y=0.15)
plt.tight_layout(); plt.savefig(FIG / "01_base_rate.png", bbox_inches="tight"); plt.show()

### 2. Where the risk signal lives: current inspection vs prior history
The forward-risk rate split by (a) whether the current inspection was itself bad, and (b) the establishment's prior bad-rate. Chicago leans on the current visit; NYC/LA lean more on the track record.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
# (a) next-risk by current-bad
for c, d in CITY.items():
    df = d["df"]
    g = df.groupby(df[d["cur"]].fillna(0).astype(int))[d["label"]].mean()
    axes[0].plot([0, 1], [g.get(0, np.nan) * 100, g.get(1, np.nan) * 100],
                 "-o", color=COL[c], label=c, lw=2)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["clean now", "bad now"])
axes[0].set_ylabel("Next-inspection risk (%)"); axes[0].set_title("By current inspection outcome")
axes[0].legend(frameon=False)
# (b) next-risk by prior bad-rate bucket
bins = [-0.01, 0.001, 0.2, 0.4, 0.6, 1.01]
labels = ["0", "0-20%", "20-40%", "40-60%", "60%+"]
for c, d in CITY.items():
    df = d["df"].copy()
    df["bkt"] = pd.cut(df[d["prior"]].fillna(0), bins=bins, labels=labels)
    g = df.groupby("bkt", observed=True)[d["label"]].mean() * 100
    axes[1].plot(range(len(g)), g.values, "-o", color=COL[c], label=c, lw=2)
axes[1].set_xticks(range(len(labels))); axes[1].set_xticklabels(labels, rotation=20)
axes[1].set_xlabel("Prior bad-rate"); axes[1].set_title("By prior track record")
axes[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "02_signal_source.png", bbox_inches="tight"); plt.show()

### 3. The NYC closure signal (the 2026-07-18 feature win)
NYC's DOHMH enforcement action (`cur_closed`) separates future risk sharply, on top of the numeric score. This is the family that moved NYC +0.02 PR-AUC. Chicago/LA have no direct equivalent field.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
d = CITY["NYC"]["df"]
g = d.groupby(d["cur_closed"].astype(int))["y_next_bc"].mean() * 100
share = d["cur_closed"].mean() * 100
bars = ax.bar(["not closed", "closed by DOHMH"], [g.get(0, np.nan), g.get(1, np.nan)],
              color=["#B8CBB0", COL["NYC"]])
for b, v in zip(bars, [g.get(0, 0), g.get(1, 0)]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f"{v:.0f}%",
            ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Next-inspection B/C rate (%)")
ax.set_title(f"NYC: a closure predicts future risk ({share:.1f}% of inspections)")
ax.margins(y=0.15)
plt.tight_layout(); plt.savefig(FIG / "03_nyc_closure.png", bbox_inches="tight"); plt.show()

### 4. Seasonality: present in Chicago, flat in NYC/LA
Forward-risk by inspection month. Chicago shows a seasonal wobble (why calendar features help it); NYC/LA are flatter (why calendar failed the gate there).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for c, d in CITY.items():
    df = d["df"]
    g = df.groupby("month")[d["label"]].mean() * 100
    g = g / g.mean()  # normalize to each city's own mean so shapes compare
    ax.plot(g.index, g.values, "-o", color=COL[c], label=c, lw=2, ms=4)
ax.axhline(1.0, color="#999", ls=":", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Inspection month"); ax.set_ylabel("Risk relative to city mean")
ax.set_title("Seasonality of forward risk (normalized per city)")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "04_seasonality.png", bbox_inches="tight"); plt.show()

### 5. Served-model performance per city
From each city's `methodology.json` (served XGB, held-out test). ROC-AUC is base-rate-independent, so it's the fairest cross-city comparison; PR-AUC tracks each city's prevalence.

In [ ]:
import json
perf = {}
for c, path in [("Chicago", "app/public/data/methodology.json"),
                ("NYC", "app/public/data/nyc/methodology.json"),
                ("LA", "app/public/data/la/methodology.json")]:
    h = json.loads((ROOT / path).read_text()).get("headline", {})
    perf[c] = {"PR-AUC": h.get("pr_auc"), "ROC-AUC": h.get("roc_auc")}
pdf = pd.DataFrame(perf).T
fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(pdf)); w = 0.38
ax.bar(x - w / 2, pdf["PR-AUC"], w, label="PR-AUC", color="#5B8FB0")
ax.bar(x + w / 2, pdf["ROC-AUC"], w, label="ROC-AUC", color="#C98A5E")
for i, c in enumerate(pdf.index):
    ax.text(i - w / 2, pdf["PR-AUC"][c] + 0.01, f"{pdf['PR-AUC'][c]:.2f}", ha="center", fontsize=9)
    ax.text(i + w / 2, pdf["ROC-AUC"][c] + 0.01, f"{pdf['ROC-AUC'][c]:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(pdf.index)
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_title("Served-model performance (held-out test)")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "05_model_performance.png", bbox_inches="tight"); plt.show()
pdf.round(3)

## Takeaways

1. **Prevalence spans ~4x** (LA lowest, NYC highest) — raw PR-AUC is not comparable across cities; use ROC-AUC and lift.
2. **The dominant signal flips**: Chicago's current inspection carries it, NYC/LA lean on the prior track record (they inspect ~annually, so "now" is staler).
3. **City-native signals matter**: NYC's closure/enforcement action is a strong, orthogonal predictor (the +0.02 PR-AUC win); Chicago/LA have no equivalent field.
4. **Seasonality is real in Chicago, flat in NYC/LA** — which is exactly why calendar features passed the gate in Chicago and failed in NYC/LA.
5. **XGBoost + Platt is the right served recipe everywhere**, but the *features* should be chosen per city from the same concept menu, not standardized to a lowest common denominator.